# Percobaan 5 — SwinV2 (5 Fold)

Inferensi 5 fold untuk SwinV2, dengan kalibrasi logit.


In [7]:
!pip install -q timm==1.0.11 scipy


**PENTING**: pip install kadang bikin numpy/pandas tidak konsisten biner. Sel di bawah restart
kernel paksa (normal, bukan crash). Setelah reconnect, lanjut dari sel import — JANGAN ulang pip.


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart paksa; lanjut dari sel berikutnya setelah reconnect


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import json
import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import timm
import timm.data
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from scipy.optimize import minimize
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__,
      "| scipy:", scipy.__version__, "| sklearn:", sklearn.__version__)
if DEVICE.type != "cuda":
    print("PERINGATAN: GPU tidak aktif -- inferensi 5 fold model di CPU akan sangat lambat. "
          "Runtime -> Change runtime type -> T4 GPU, lalu Run all ulang.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda | torch: 2.11.0+cu128 | timm: 1.0.11 | scipy: 1.16.3 | sklearn: 1.6.1


## Config

`available_folds` HARUS cocok dengan fold yang benar-benar punya `best.ckpt` di Drive. `drive_folder`
HARUS persis sama nama foldernya (spasi/kapital) dengan yang ada di `Training_Output/`.


In [11]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"
TEMPLATE_PATH = DRIVE_ROOT / "BDC2026" / "submission.csv"
TRAIN_PROCESSED_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "processed"
FOLD_ASSIGNMENT_PATH = DRIVE_ROOT / "Preprocessing_Output" / "manifests" / "fold_assignment.csv"
CKPT_ROOT = DRIVE_ROOT / "Training_Output"
TEST_IMAGE_EXT = "jpg"

OUTPUT_ROOT = DRIVE_ROOT / "Percobaan Submit" / "Percobaan 5"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_TAG = "Percobaan 5"

# TEMPLATE_PATH SENGAJA tidak diwajibkan ada: folder "Big Data Challenge..." ada di "Dibagikan
# kepada saya" (shared), bukan My Drive. Colab bisa menembus ke SUBFOLDER shared (mis. BDC2026/test)
# tapi kadang tidak konsisten untuk FILE tunggal milik orang lain (submission.csv) walau filenya ADA.
# Daftar id test direkonstruksi dari isi folder test kalau template tidak terbaca (lihat sel di bawah).
for _p in [TEST_ROOT, TRAIN_PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH, CKPT_ROOT]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} tidak ketemu -- cek Drive ter-mount & path benar.")

# ARCH_NAME -> spesifikasi. Urutan dict = urutan arsitektur yang dipakai konsisten di seluruh notebook.
ARCH_REGISTRY = {
    "swinv2_base": dict(
        timm_tag="swinv2_base_window12to24_192to384.ms_in22k_ft_in1k", drop_path_rate=0.3,
        drive_folder="Swin Transformer V2", available_folds=[0, 1, 2, 3, 4],
    ),
}
ARCHS = list(ARCH_REGISTRY.keys())

# irisan fold yang SEMUA arsitektur punya -> dipakai untuk optimasi bobot & kalibrasi (tanpa bocor)
COMMON_FOLDS = sorted(set.intersection(*[set(s["available_folds"]) for s in ARCH_REGISTRY.values()]))
print("Fold irisan (dipakai optimasi bobot & kalibrasi):", COMMON_FOLDS)

# cek semua checkpoint yang dibutuhkan ada
for arch, spec in ARCH_REGISTRY.items():
    for k in spec["available_folds"]:
        p = CKPT_ROOT / spec["drive_folder"] / f"fold{k}" / "best.ckpt"
        if not p.exists():
            raise FileNotFoundError(f"{p} tidak ketemu -- cek drive_folder/available_folds di ARCH_REGISTRY.")
total_models = sum(len(s["available_folds"]) for s in ARCH_REGISTRY.values())
print(f"Total checkpoint model: {total_models} (dipakai untuk inferensi test)")


Fold irisan (dipakai optimasi bobot & kalibrasi): [0, 1, 2, 3, 4]
Total checkpoint model: 5 (dipakai untuk inferensi test)


## Helper: transform eval, TTA, pembuatan model (byte-identical logika dengan NB04)


In [12]:
def build_eval_transform(img_size, mean, std):
    resize_short_side = round(img_size / 0.95)
    return T.Compose([
        T.Resize(resize_short_side, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


@torch.no_grad()
def tta_predict_probs(model, images):
    images = images.to(DEVICE, non_blocking=True)
    probs_identity = F.softmax(model(images), dim=1)
    probs_flip = F.softmax(model(torch.flip(images, dims=[3])), dim=1)
    return (probs_identity + probs_flip) / 2.0


def create_inference_model(timm_tag, drop_path_rate, img_size):
    kwargs = dict(pretrained=False, num_classes=NUM_CLASSES, drop_path_rate=drop_path_rate, img_size=img_size)
    try:
        model = timm.create_model(timm_tag, **kwargs)
    except TypeError:
        kwargs.pop("img_size")
        model = timm.create_model(timm_tag, **kwargs)
    return model


def resolve_norm_stats(model):
    data_cfg = timm.data.resolve_model_data_config(model)
    return list(data_cfg["mean"]), list(data_cfg["std"])


class ImageIdDataset(Dataset):
    def __init__(self, image_ids, root, transform):
        self.image_ids = np.asarray(image_ids).astype(np.int64)
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        with Image.open(f"{self.root}/{image_id}.{TEST_IMAGE_EXT}") as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        return tensor, image_id


## Salin gambar yang diperlukan ke disk lokal (hindari bottleneck Drive)

Yang diperlukan: gambar train di fold irisan (untuk OOF) + seluruh gambar test.


In [13]:
LOCAL_STAGING = Path("/content/local_staging")
LOCAL_PROCESSED = LOCAL_STAGING / "processed"
LOCAL_TEST = LOCAL_STAGING / "test"
LOCAL_PROCESSED.mkdir(parents=True, exist_ok=True)
LOCAL_TEST.mkdir(parents=True, exist_ok=True)

N_WORKERS = 64


def parallel_copy(pairs, desc, n_workers=N_WORKERS, per_file_timeout=60):
    def _one(src, dest):
        dest = Path(dest)
        if dest.exists() and dest.stat().st_size > 0:
            return
        shutil.copy2(src, dest)
    errors = []
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_one, s, d): (s, d) for s, d in pairs}
        for fut in tqdm(list(futs), total=len(futs), desc=desc):
            s, d = futs[fut]
            try:
                fut.result(timeout=per_file_timeout)
            except Exception as e:
                errors.append({"src": str(s), "dest": str(d), "error": str(e)})
    return errors


fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
common_rows = fold_assignment[fold_assignment["fold"].isin(COMMON_FOLDS)].sort_values("image_id").reset_index(drop=True)
common_ids = common_rows["image_id"].values
y_true = common_rows["label"].values.astype(np.int64)
print(f"Gambar di fold irisan {COMMON_FOLDS}: {len(common_ids)} (dipakai optimasi bobot & kalibrasi)")

# Template id: coba baca file resmi dulu; kalau tidak terbaca (quirk shared-file di Colab),
# rekonstruksi dari isi folder test. Urutan resmi = 1..1458 berurutan (soal: "diurutkan 1 hingga
# 1458"), jadi rekonstruksi ini identik urutannya dengan template & tetap sah.
def get_template_ids():
    try:
        df = pd.read_csv(TEMPLATE_PATH)
        ids = df["id"].values
        if len(ids) == 1458:
            print(f"Template dibaca dari file resmi: {TEMPLATE_PATH}")
            return ids
        print(f"Template file terbaca tapi barisnya {len(ids)} (bukan 1458) -- fallback ke listing folder test.")
    except Exception as e:
        print(f"Template file tidak terbaca ({e}) -- fallback: rekonstruksi id dari isi folder test.")
    stems = sorted((int(p.stem) for p in TEST_ROOT.glob(f"*.{TEST_IMAGE_EXT}")))
    ids = np.array(stems, dtype=np.int64)
    assert set(ids.tolist()) == set(range(1, 1459)), (
        f"id folder test != 1..1458 (dapat {len(ids)} file, min={ids.min()}, max={ids.max()}) -- "
        f"stop, jangan submit urutan yang belum tentu cocok template."
    )
    print(f"Template direkonstruksi dari folder test: {len(ids)} id (1..1458, berurutan).")
    return ids


template_ids = get_template_ids()
assert len(template_ids) == 1458, f"template harus 1458 baris, dapat {len(template_ids)}"

parallel_copy([(TRAIN_PROCESSED_ROOT / f"{i}.jpg", LOCAL_PROCESSED / f"{i}.jpg") for i in common_ids],
              desc="salin train (fold irisan)")
parallel_copy([(TEST_ROOT / f"{i}.jpg", LOCAL_TEST / f"{i}.jpg") for i in template_ids],
              desc="salin test")

# Cache hasil inferensi di DRIVE (bukan /content lokal, supaya selamat kalau kernel mati/restart).
# Tiap (arsitektur, fold) disimpan begitu selesai -> run ulang otomatis skip yang sudah ada.
CACHE_DIR = DRIVE_ROOT / "Preprocessing_Output" / "percobaan5_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Staging selesai. Cache inferensi di:", CACHE_DIR)


Gambar di fold irisan [0, 1, 2, 3, 4]: 26448 (dipakai optimasi bobot & kalibrasi)
Template dibaca dari file resmi: /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/BDC2026/submission.csv


salin train (fold irisan):   0%|          | 0/26448 [00:00<?, ?it/s]

salin test:   0%|          | 0/1458 [00:00<?, ?it/s]

Staging selesai. Cache inferensi di: /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Preprocessing_Output/percobaan5_cache


## Regenerasi OOF di fold irisan (untuk optimasi bobot & kalibrasi)

Untuk tiap arsitektur, untuk tiap fold irisan k: muat `best.ckpt` fold-k (model yang TIDAK melatih
gambar fold-k), prediksi gambar validasi fold-k dengan TTA. Ini = prediksi OOF sah (tanpa bocor).


In [14]:
def infer_fold_on_ids(spec, k, image_ids, root):
    # muat best.ckpt fold-k, prediksi image_ids dengan TTA -> (probs sejajar image_ids, )
    best = torch.load(CKPT_ROOT / spec["drive_folder"] / f"fold{k}" / "best.ckpt", map_location="cpu")
    img_size = best["img_size"]
    model = create_inference_model(spec["timm_tag"], spec["drop_path_rate"], img_size).to(DEVICE)
    model.load_state_dict(best["weights"])
    model.eval()
    mean, std = resolve_norm_stats(model)
    loader = DataLoader(ImageIdDataset(image_ids, root, build_eval_transform(img_size, mean, std)),
                         batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    id_to_local = {int(v): i for i, v in enumerate(image_ids)}
    out = np.zeros((len(image_ids), NUM_CLASSES), dtype=np.float64)
    for images, ids in tqdm(loader, desc=f"{spec['drive_folder']} fold{k}", leave=False):
        probs = tta_predict_probs(model, images).cpu().numpy()
        for image_id, p in zip(ids.numpy(), probs):
            out[id_to_local[int(image_id)]] = p
    del model
    torch.cuda.empty_cache()
    return out


row_of_id = {int(i): r for r, i in enumerate(common_ids)}  # image_id -> baris di matriks OOF
arch_oof_probs = {arch: np.zeros((len(common_ids), NUM_CLASSES), dtype=np.float64) for arch in ARCHS}

for arch in ARCHS:
    spec = ARCH_REGISTRY[arch]
    for k in COMMON_FOLDS:
        val_ids = common_rows[common_rows["fold"] == k]["image_id"].values
        cache = CACHE_DIR / f"oof_{arch}_fold{k}.npz"
        if cache.exists():
            data = np.load(cache)
            fold_ids, fold_probs = data["ids"], data["probs"]
            print(f"[cache] OOF {arch} fold{k} dimuat dari Drive")
        else:
            fold_probs = infer_fold_on_ids(spec, k, val_ids, LOCAL_PROCESSED)
            fold_ids = np.asarray(val_ids, dtype=np.int64)
            np.savez(cache, ids=fold_ids, probs=fold_probs)
            print(f"[hitung] OOF {arch} fold{k} selesai & disimpan ke cache")
        for image_id, p in zip(fold_ids, fold_probs):
            arch_oof_probs[arch][row_of_id[int(image_id)]] = p

probs_by_arch = [arch_oof_probs[a] for a in ARCHS]
oof_fold = common_rows["fold"].values
print("Semua OOF fold irisan siap.")


Swin Transformer V2 fold0:   0%|          | 0/166 [00:00<?, ?it/s]

[hitung] OOF swinv2_base fold0 selesai & disimpan ke cache


Swin Transformer V2 fold1:   0%|          | 0/166 [00:00<?, ?it/s]

[hitung] OOF swinv2_base fold1 selesai & disimpan ke cache


Swin Transformer V2 fold2:   0%|          | 0/166 [00:00<?, ?it/s]

[hitung] OOF swinv2_base fold2 selesai & disimpan ke cache


Swin Transformer V2 fold3:   0%|          | 0/166 [00:00<?, ?it/s]

[hitung] OOF swinv2_base fold3 selesai & disimpan ke cache


Swin Transformer V2 fold4:   0%|          | 0/166 [00:00<?, ?it/s]

[hitung] OOF swinv2_base fold4 selesai & disimpan ke cache
Semua OOF fold irisan siap.


## §10.2 Optimasi bobot ensemble — guarded weighted average (logika NB03)

`w = softmax(z)`, `z` dioptimasi Nelder-Mead (5 restart) untuk maksimalkan OOF Macro-F1. Guardrail
distabilitas disesuaikan ke fold irisan yang tersedia.


In [15]:
def softmax_np(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def ensemble_probs(weights, probs_list):
    return sum(w * p for w, p in zip(weights, probs_list))


def neg_macro_f1_for_z(z, probs_list, y):
    w = softmax_np(z)
    return -f1_score(y, ensemble_probs(w, probs_list).argmax(axis=1), average="macro")


def search_weights(probs_list, y, n_restarts=5, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(probs_list)
    best = None
    for i in range(n_restarts):
        z0 = np.zeros(n) if i == 0 else rng.normal(scale=0.5, size=n)
        res = minimize(neg_macro_f1_for_z, z0, args=(probs_list, y), method="Nelder-Mead",
                        options={"xatol": 1e-4, "fatol": 1e-6, "maxiter": 2000})
        if best is None or res.fun < best.fun:
            best = res
    return softmax_np(best.x), -best.fun


optimized_weights, optimized_f1 = search_weights(probs_by_arch, y_true)
equal_weights = np.full(len(ARCHS), 1.0 / len(ARCHS))
equal_f1 = f1_score(y_true, ensemble_probs(equal_weights, probs_by_arch).argmax(axis=1), average="macro")

print("bobot optimized:", dict(zip(ARCHS, optimized_weights.round(4))), "-> OOF macro-F1 =", round(optimized_f1, 5))
print("bobot equal:    ", dict(zip(ARCHS, equal_weights.round(4))), "-> OOF macro-F1 =", round(equal_f1, 5))


bobot optimized: {'swinv2_base': np.float64(1.0)} -> OOF macro-F1 = 0.98543
bobot equal:     {'swinv2_base': np.float64(1.0)} -> OOF macro-F1 = 0.98543


In [16]:
# Guardrail 1 — stability (leave-one-fold-out atas fold irisan yang ada)
loFo = []
for k in COMMON_FOLDS:
    mask = oof_fold != k
    w_k, _ = search_weights([p[mask] for p in probs_by_arch], y_true[mask], n_restarts=3, seed=SEED + k)
    loFo.append(w_k)
loFo = np.array(loFo)
cv_per_arch = loFo.std(axis=0) / np.clip(loFo.mean(axis=0), 1e-8, None)
guardrail_1 = bool((cv_per_arch <= 0.3).all())
print("CV bobot leave-one-fold-out:", dict(zip(ARCHS, cv_per_arch.round(4))),
      "->", "LOLOS" if guardrail_1 else "GAGAL")
print(f"(catatan: cuma {len(COMMON_FOLDS)} fold irisan, jadi guardrail stabilitas ini lebih lemah dari NB03 5-fold)")

# Guardrail 2 — minimum-gain floor
guardrail_2 = bool((optimized_f1 - equal_f1) >= 0.002)
print(f"gain optimized vs equal = {optimized_f1 - equal_f1:.5f} ->", "LOLOS" if guardrail_2 else "GAGAL")

# Guardrail 3 — per-arch floor
low_weight = [a for a, w in zip(ARCHS, optimized_weights) if w < 0.05]
guardrail_3 = bool(len(low_weight) == 0)
print(f"arsitektur bobot < 0.05: {low_weight} ->", "LOLOS" if guardrail_3 else "GAGAL")

all_pass = guardrail_1 and guardrail_2 and guardrail_3
if all_pass:
    final_weights, final_scheme = optimized_weights, "optimized"
else:
    final_weights, final_scheme = equal_weights, "equal"
print(f"\nSkema bobot FINAL: {final_scheme} -> {dict(zip(ARCHS, final_weights.round(4)))}")
print("(kalibrasi tetap diterapkan apa pun skema bobotnya -- itu sumber perbaikan utama vs percobaan 1)")


CV bobot leave-one-fold-out: {'swinv2_base': np.float64(0.0)} -> LOLOS
(catatan: cuma 5 fold irisan, jadi guardrail stabilitas ini lebih lemah dari NB03 5-fold)
gain optimized vs equal = 0.00000 -> GAGAL
arsitektur bobot < 0.05: [] -> LOLOS

Skema bobot FINAL: equal -> {'swinv2_base': np.float64(1.0)}
(kalibrasi tetap diterapkan apa pun skema bobotnya -- itu sumber perbaikan utama vs percobaan 1)


## §10.3 Kalibrasi = post-hoc logit adjustment (SEKALI, di atas ensemble berbobot)

`argmax_c[log(p_ens,c) + delta_c]`, `delta_0 := 0`; grid kasar lalu refinement Nelder-Mead. Aturan
konservatif: di antara solusi selisih F1 <= 0.1% dari maksimum, pilih `||delta||` terkecil.


In [17]:
ens_oof = ensemble_probs(final_weights, probs_by_arch)


def apply_logit_adjustment(probs, delta):
    return (np.log(np.clip(probs, 1e-12, 1.0)) + delta).argmax(axis=1)


def macro_f1_for_delta(delta12, probs, y):
    delta = np.array([0.0, delta12[0], delta12[1]])
    return f1_score(y, apply_logit_adjustment(probs, delta), average="macro")


def search_calibration(probs, y, grid_step=0.05, bound=1.5, tol=0.001):
    grid = np.round(np.arange(-bound, bound + 1e-9, grid_step), 4)
    cands = []
    for d1 in grid:
        for d2 in grid:
            cands.append((macro_f1_for_delta((d1, d2), probs, y), float(np.hypot(d1, d2)), (float(d1), float(d2))))
    coarse = max(cands, key=lambda c: c[0])
    ref = minimize(lambda d: -macro_f1_for_delta(d, probs, y), x0=np.array(coarse[2]),
                    method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-6})
    rd = tuple(np.clip(ref.x, -bound, bound).tolist())
    cands.append((macro_f1_for_delta(rd, probs, y), float(np.hypot(*rd)), rd))
    mx = max(c[0] for c in cands)
    near = sorted([c for c in cands if c[0] >= mx - tol], key=lambda c: c[1])
    _, _, chosen = near[0]
    return np.array([0.0, chosen[0], chosen[1]]), macro_f1_for_delta(chosen, probs, y)


delta, f1_after = search_calibration(ens_oof, y_true)
f1_before = f1_score(y_true, ens_oof.argmax(axis=1), average="macro")
print(f"OOF macro-F1 (fold irisan) sebelum kalibrasi: {f1_before:.5f}")
print(f"OOF macro-F1 (fold irisan) sesudah kalibrasi: {f1_after:.5f}  (delta={delta.round(4).tolist()})")
print(f"\nSebagai pembanding: bobot equal TANPA kalibrasi (setara percobaan 1) = {equal_f1:.5f}")


OOF macro-F1 (fold irisan) sebelum kalibrasi: 0.98543
OOF macro-F1 (fold irisan) sesudah kalibrasi: 0.98543  (delta=[0.0, 0.0, 0.0])

Sebagai pembanding: bobot equal TANPA kalibrasi (setara percobaan 1) = 0.98543


## Diagnostik — seberapa luas bias "ditarik ke kelas Organic" di OOF?

Kalibrasi di atas menemukan delta=0 (tidak ada koreksi yang membantu di OOF fold irisan). Ini cuma
membuktikan bias itu TIDAK KELIHATAN di OOF train -- belum tentu berarti bias itu TIDAK ADA/SEMPIT.
Sel ini mengukur langsung: dari gambar yang label aslinya BUKAN Organic, berapa persen yang
diprediksi Organic (false-Organic rate). Kalau kecil (<1%) -> bias ini sempit/khusus, gap ke 99
kemungkinan besar soal kelengkapan model (20 fold). Kalau besar -> bias ini sistematis luas, perlu
diperbaiki lewat augmentasi (naikkan color_jitter/grayscale) & retrain, BUKAN cuma menunggu fold.


In [18]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_oof = apply_logit_adjustment(ens_oof, delta)
label_names = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

print(classification_report(y_true, y_pred_oof, target_names=[label_names[i] for i in range(3)], digits=4))

cm = confusion_matrix(y_true, y_pred_oof)
print("Confusion matrix OOF (baris=label asli, kolom=prediksi):")
print(pd.DataFrame(cm, index=list(label_names.values()), columns=list(label_names.values())))

non_organic_mask = y_true != 2
n_false_organic = int((y_pred_oof[non_organic_mask] == 2).sum())
false_organic_rate = n_false_organic / non_organic_mask.sum()
print(f"\nDari {int(non_organic_mask.sum())} gambar OOF berlabel asli BUKAN Organic, "
      f"{n_false_organic} ({false_organic_rate:.2%}) salah diprediksi Organic.")
if false_organic_rate < 0.01:
    print("-> Bias di OOF SEMPIT. Konsisten dengan dugaan: kasus bermasalah itu khusus/langka di "
          "train, bukan pola sistematis luas. Menunggu 20 model (5 fold) masuk akal, tapi jangan "
          "berharap ini sendirian menutup gap ke 99 -- kemungkinan perlu tambahan augmentasi juga.")
else:
    print("-> Bias di OOF CUKUP LUAS. Ini sinyal kuat bahwa augmentasi (naikkan color_jitter_scale, "
          "tambah probabilitas grayscale) & retrain diperlukan -- menunggu fold saja TIDAK CUKUP.")


              precision    recall  f1-score   support

  Recyclable     0.9733    0.9865    0.9798      9979
  Electronic     0.9896    0.9952    0.9924      3929
     Organic     0.9903    0.9779    0.9841     12540

    accuracy                         0.9837     26448
   macro avg     0.9844    0.9865    0.9854     26448
weighted avg     0.9838    0.9837    0.9837     26448

Confusion matrix OOF (baris=label asli, kolom=prediksi):
            Recyclable  Electronic  Organic
Recyclable        9844          18      117
Electronic          16        3910        3
Organic            254          23    12263

Dari 13908 gambar OOF berlabel asli BUKAN Organic, 120 (0.86%) salah diprediksi Organic.
-> Bias di OOF SEMPIT. Konsisten dengan dugaan: kasus bermasalah itu khusus/langka di train, bukan pola sistematis luas. Menunggu 20 model (5 fold) masuk akal, tapi jangan berharap ini sendirian menutup gap ke 99 -- kemungkinan perlu tambahan augmentasi juga.


## Inferensi test — semua fold yang tersedia per arsitektur -> bobot final -> kalibrasi


In [19]:
arch_test_probs = {}

for arch in ARCHS:
    spec = ARCH_REGISTRY[arch]
    fold_probs_list = []
    for k in spec["available_folds"]:
        cache = CACHE_DIR / f"test_{arch}_fold{k}.npz"
        if cache.exists():
            probs = np.load(cache)["probs"]  # sudah sejajar urutan template_ids
            print(f"[cache] test {arch} fold{k} dimuat dari Drive")
        else:
            probs = infer_fold_on_ids(spec, k, template_ids, LOCAL_TEST)  # sejajar template_ids
            np.savez(cache, probs=probs)
            print(f"[hitung] test {arch} fold{k} selesai & disimpan ke cache")
        fold_probs_list.append(probs)
    arch_test_probs[arch] = np.mean(fold_probs_list, axis=0)
    print(f"{arch}: inferensi test siap ({len(spec['available_folds'])} fold)")

ens_test = ensemble_probs(final_weights, [arch_test_probs[a] for a in ARCHS])
final_preds = apply_logit_adjustment(ens_test, delta)


Swin Transformer V2 fold0:   0%|          | 0/46 [00:00<?, ?it/s]

[hitung] test swinv2_base fold0 selesai & disimpan ke cache


Swin Transformer V2 fold1:   0%|          | 0/46 [00:00<?, ?it/s]

[hitung] test swinv2_base fold1 selesai & disimpan ke cache


Swin Transformer V2 fold2:   0%|          | 0/46 [00:00<?, ?it/s]

[hitung] test swinv2_base fold2 selesai & disimpan ke cache


Swin Transformer V2 fold3:   0%|          | 0/46 [00:00<?, ?it/s]

[hitung] test swinv2_base fold3 selesai & disimpan ke cache


Swin Transformer V2 fold4:   0%|          | 0/46 [00:00<?, ?it/s]

[hitung] test swinv2_base fold4 selesai & disimpan ke cache
swinv2_base: inferensi test siap (5 fold)


## Validation gate + tulis submission


In [20]:
submission_df = pd.DataFrame({"id": template_ids, "predicted": final_preds})

assert len(submission_df) == 1458
assert np.array_equal(submission_df["id"].values, template_ids), "urutan baris menyimpang dari template"
assert submission_df["predicted"].isin([0, 1, 2]).all()
assert not submission_df.isna().any().any()
print("Validation gate LOLOS.")

output_path = OUTPUT_ROOT / f"submission_{SUBMISSION_TAG}.csv"
submission_df.to_csv(output_path, index=False)
print(f"Menulis {output_path}")

# simpan juga config untuk audit trail
audit = {
    "archs": ARCHS,
    "available_folds": {a: ARCH_REGISTRY[a]["available_folds"] for a in ARCHS},
    "common_folds_for_tuning": COMMON_FOLDS,
    "final_scheme": final_scheme,
    "final_weights": dict(zip(ARCHS, final_weights.tolist())),
    "optimized_weights": dict(zip(ARCHS, optimized_weights.tolist())),
    "calibration_delta": delta.tolist(),
    "oof_macro_f1_before_calibration": float(f1_before),
    "oof_macro_f1_after_calibration": float(f1_after),
    "oof_macro_f1_equal_no_calibration": float(equal_f1),
    "guardrails": {"stability": guardrail_1, "min_gain": guardrail_2, "per_arch_floor": guardrail_3, "all_pass": all_pass},
}
with open(OUTPUT_ROOT / f"audit_percobaan2_{SUBMISSION_TAG}.json", "w") as f:
    json.dump(audit, f, indent=2)

test_dist = submission_df["predicted"].value_counts(normalize=True).sort_index().to_dict()
print("distribusi prediksi test:", {k: round(v, 4) for k, v in test_dist.items()})
print("audit tersimpan:", OUTPUT_ROOT / f"audit_percobaan2_{SUBMISSION_TAG}.json")


Validation gate LOLOS.
Menulis /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Percobaan Submit/Percobaan 5/submission_Percobaan 5.csv
distribusi prediksi test: {0: 0.3477, 1: 0.1392, 2: 0.513}
audit tersimpan: /content/drive/MyDrive/Big Data Challenge - Satria Data 2026/Percobaan Submit/Percobaan 5/audit_percobaan2_Percobaan 5.json


## Catatan

Submission ini SAH (murni model + kalibrasi, nol prediksi manual). Percobaan ini khusus menguji arsitektur tunggal **SwinV2 (5 Fold)** untuk melihat performa baseline model ini sebelum di-ensemble.

Kelola jatah submission: masih ada 2 dari 3. Gunakan sebijak mungkin.